# Setup

## Imports

In [1]:
from pathlib import Path
import os
import pandas as pd
import sys
from matplotlib import pyplot as plt
import seaborn as sns
from adjustText import adjust_text
from matplotlib.patches import Patch
from matplotlib.ticker import LogLocator, NullFormatter
import numpy as np



import hatfield_funcs as hf

# project root = one level up from notebooks so maybe more here
project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root / "src"))

from biofilm_spectra.processing.spectral_matrix import to_spectral_matrix

from biofilm_spectra.processing.micasense import *
from my_extra_to_hatfield_funcs.io_funcs import *

In [2]:
plt.style.use("default")
%config InlineBackend.figure_format = 'retina'

In [3]:
CONTAINER_NAME = "wesa-fup-restricted-027817p"

if os.getenv("GA_STORAGE_ACCOUNT_NAME") is not None:
    # this is what gets used when using Azure cloud storage (GeoAnalytics)
    hf.config.configure_azure()
    hf.config.configure_data_root_path(CONTAINER_NAME)
else:
    raise Exception(
        "This notebook is configured to run in the cloud. "
        "Please set up your Azure storage account and container name."
    )


## Parameters

In [4]:
SITES = ["brunswick", "westham"]
SITE_COLORS = {
    "brunswick": "#0455FE",
    "westham": "#56E00A",
}
SITE_LABELS = {
    "brunswick": "Impact Area",
    "westham": "Westham Control Area",
}
SITE_SHORTLABELS = {
    "brunswick": "IA",
    "westham": "WI",
}

PIGMENT_LIST = [
    "Peri",
    "Chlc2",
    "Fuco",
    "Ddx",
    "Allo",
    "Myxo",
    "Lut",
    "Zea",
    "Cant",
    "Chlb",
    "Chl_allo1",
    "Chl_allo2",
    "Chla",
    "DVCh-a",
    "Echi",
    "Phtb",
    "Phta",
    "Acar",
    "Beta",
]
FA_LIST = [
    "C6",
    "C8",
    "C10",
    "C11",
    "C12",
    "C13",
    "C14:0",  # Corrected from C14-0 to C14:0
    "C14:1",  # Corrected from C14-1 to C14:1
    "C15",
    "C15:1", # Corrected from C15-1 to C15:1
    "C16",
    "C16:1", # Corrected from C16-1 to C16:1
    "C17:0",
    "C17:1",
    "C18:0",
    "C18:1N9T",
    "C18:1N9C",
    "C18:2N6T",
    "C18:2N6C",
    "C20:0",
    "C18:3N6",
    "C20:1N9",
    "C18:3N3",
    "C21:0",
    "C20:2",
    "C22:0",
    "C20:3N6",
    "C22:1N9",
    "C20:3N3",
    "C23:0",
    "C20:4N6",
    "C22:2",
    "C24:0",
    "C20:5N3",
    "C24:1N9",
    "C22:6N3",
]

NORMALIZING_CHLA_COLUMN = "Chla + allomers"  #'Chla' #"Chla + DVCh-a + allomers" # Defined in the notebook to preprocess lab data

In [5]:
# Input paths
path_to_combined_dataset = Path("data/processed/full_dataset_combined_grouped.csv")


In [6]:
# Output path
path_output_dir = Path('./outputs/lab_data')
path_output_dir.mkdir(parents=True, exist_ok=True)

# Read data

In [7]:
df_combined_grouped = df_test = read_df_from_csv(
    path_to_combined_dataset,
    index_col=[0, 1, 2, 3],
    header=[0, 1, 2],
)

df_combined_grouped_R = df_combined_grouped.xs("R", level="sample_type")
df_combined_R = df_combined_grouped_R.copy()
df_combined_R.columns = df_combined_grouped_R.columns.get_level_values(2)

In [12]:
group_columns = df_combined_grouped_R.columns.get_level_values(0).unique()
for i_group, group_column in enumerate(group_columns):
    print(group_column)
    subgroup_columns = df_combined_grouped_R.xs(group_column, level=0, axis=1).columns.get_level_values(0).unique()
    for subgroup_column in subgroup_columns:
        print('\t', subgroup_column)
        if i_group==0:
            columns = df_combined_grouped_R.xs((group_column, subgroup_column), level=(0, 1), axis=1).columns.get_level_values(0).unique()
            for column in columns:
                print('\t\t', column)

Contextual
	 Mass-related
		 Sample Total Dry Mass (g)
		 Extraction Dry Mass (mg)
		 Area Sampled (m2)
		 /g to /m2 Conversion Factor (g/m2)
	 Composition
		 Moisture (%)
		 Percent Sand (%)
		 Percent Silt (%)
		 Percent Clay (%)
	 Time-related
		 Sample Date
		 Time of Emergence (decimal)
		 Time of Acquisition (decimal)
		 Time of Sampling After Emergence (decimal)
		 Accumulated PAR Between Emergence and Acquisition (mol/m2)
Pigments
	 Lab Concentrations (/g)
	 Surface Densities (/m2)
	 Normalized to Chla + allomers
	 Ratios
Fatty Acids
	 Lab Concentrations (/g)
	 Surface Densities (/m2)
	 Normalized to Chla + allomers
	 Ratios
Nutritional
	 Lab Concentrations (/g)
	 Surface Densities (/m2)
	 Normalized to Chla + allomers
Spectral
	 Bands
	 Indices


In [23]:
lab_unit_subgroup = 'Lab Concentrations (/g)' # or 'Surface Densities (/m2)'

columns_to_test_outliers = (
    [
        'Sample Total Dry Mass (g)',
        'Moisture (%)',
        'Percent Sand (%)',
    ]
    + list(df_combined_grouped_R.xs(('Pigments', lab_unit_subgroup), level=(0, 1), axis=1).columns)
    + list(df_combined_grouped_R.xs(('Fatty Acids', lab_unit_subgroup), level=(0, 1), axis=1).columns)
    + list(df_combined_grouped_R.xs(('Nutritional', lab_unit_subgroup), level=(0, 1), axis=1).columns)
)

print(columns_to_test_outliers)


df_to_test_outliers = df_combined_R.filter(columns_to_test_outliers)
df_to_test_outliers.head()

['Sample Total Dry Mass (g)', 'Moisture (%)', 'Percent Sand (%)', 'Peri (mg/g)', 'Chlc2 (mg/g)', 'Fuco (mg/g)', 'Ddx (mg/g)', 'Allo (mg/g)', 'Myxo (mg/g)', 'Lut (mg/g)', 'Zea (mg/g)', 'Cant (mg/g)', 'Chlb (mg/g)', 'Chl_allo1 (mg/g)', 'Chl_allo2 (mg/g)', 'Chla (mg/g)', 'DVCh-a (mg/g)', 'Echi (mg/g)', 'Phtb (mg/g)', 'Phta (mg/g)', 'Acar (mg/g)', 'Beta (mg/g)', 'Chla + allomers (mg/g)', 'Chla + allomers + DVCh-a (mg/g)', 'Chla + allomers + Phta (mg/g)', 'Chla + Phta (mg/g)', 'Chla + DVCh-a (mg/g)', 'Total Chla (mg/g)', 'C6 (mg/g)', 'C8 (mg/g)', 'C10 (mg/g)', 'C11 (mg/g)', 'C12 (mg/g)', 'C13 (mg/g)', 'C14:0 (mg/g)', 'C14:1 (mg/g)', 'C15 (mg/g)', 'C15:1 (mg/g)', 'C16 (mg/g)', 'C16:1 (mg/g)', 'C17:0 (mg/g)', 'C17:1 (mg/g)', 'C18:0 (mg/g)', 'C18:1N9T (mg/g)', 'C18:1N9C (mg/g)', 'C18:2N6T (mg/g)', 'C18:2N6C (mg/g)', 'C20:0 (mg/g)', 'C18:3N6 (mg/g)', 'C20:1N9 (mg/g)', 'C18:3N3 (mg/g)', 'C21:0 (mg/g)', 'C20:2 (mg/g)', 'C22:0 (mg/g)', 'C20:3N6 (mg/g)', 'C22:1N9 (mg/g)', 'C20:3N3 (mg/g)', 'C23:0 (

Sample Total Dry Mass (g)  Moisture (%)  \
site      station_id round                                            
brunswick B1         1                     124.920787     55.812630   
                     2                     102.318373     46.904693   
          B10        1                     113.372704     53.288243   
                     2                     135.840106     45.619801   
          B11        1                      68.488373     60.344183   

                            Percent Sand (%)  Peri (mg/g)  Chlc2 (mg/g)  \
site      station_id round                                                
brunswick B1         1             91.182365     0.000130      0.000126   
                     2             73.170732     0.000207      0.000324   
          B10        1             73.684211     0.000159      0.000414   
                     2             86.597938     0.000217      0.000197   
          B11        1             71.232877     0.000083      0.000451   

                            Fuco (mg/g)  Ddx (mg/g)  Allo (mg/g)  Myxo (mg/g)  \
site      station_id round                                                      
brunswick B1         1         0.004335    0.003132     0.000217     0.000186   
                     2         0.005283    0.004532     0.000166     0.000192   
          B10        1         0.005671    0.003579     0.000381     0.000401   
                     2         0.004817    0.003055     0.000188     0.000174   
          B11        1         0.006007    0.004110     0.000346     0.000399   

                            Lut (mg/g)  ...  Total SFA (mg/g)  \
site      station_id round              ...                     
brunswick B1         1        0.000786  ...          0.230625   
                     2        0.001309  ...          0.234236   
          B10        1        0.001142  ...          0.283961   
                     2        0.000865  ...          0.152772   
          B11        1        0.001323  ...          0.195477   

                            Total MUFA (mg/g)  Total PUFA (mg/g)  \
site      station_id round                                         
brunswick B1         1               0.202373           0.054581   
                     2               0.169972           0.085933   
          B10        1               0.182341           0.041562   
                     2               0.102281           0.015296   
          B11        1               0.140141           0.023291   

                            Total Omega-3 PUFA (mg/g)  \
site      station_id round                              
brunswick B1         1                       0.035872   
                     2                       0.058168   
          B10        1                       0.026194   
                     2                       0.009579   
          B11        1                       0.014526   

                            Total Omega-6 PUFA (mg/g)  Total LC-PUFA (mg/g)  \
site      station_id round                                                    
brunswick B1         1                       0.017012              0.045470   
                     2                       0.024665              0.071340   
          B10        1                       0.013596              0.034348   
                     2                       0.004722              0.012041   
          B11        1                       0.007711              0.019056   

                            Total FA (mg/g)  Total Carbs (mg/g)  \
site      station_id round                                        
brunswick B1         1             0.487580            6.403316   
                     2             0.490141           16.955452   
          B10        1             0.507865           21.739367   
                     2             0.270348           17.564301   
          B11        1             0.358908           17.219952   

                            Caloric (kJ/g)  Proteins (mg/g)  
site      station